# Axolotl DPO — Qwen3-4B & Qwen3-14B

**Stage 2** of the SFT -> DPO pipeline for models not supported by Tinker RL.

Prerequisites: Run `axolotl_sft_qlora.ipynb` first to get an SFT checkpoint.

Preferences from Supermemory:
- PO LR: 2x-200x smaller than SFT (so ~1.5e-8 to 5e-6)
- Beta 0.1+ controls reference adherence
- DPO is simpler than RLHF (no reward model, no PPO)

## 1. Setup

In [ ]:
!nvidia-smi
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
!pip install -q axolotl[flash-attn] peft bitsandbytes wandb huggingface_hub trl
!pip install -q flash-attn --no-build-isolation

In [ ]:
from huggingface_hub import login
login()

import wandb
wandb.login()

## 2. DPO Configs

DPO requires a preference dataset with chosen/rejected pairs.

Key hyperparameters:
- `beta`: KL penalty controlling how far the model can deviate from reference (0.1 is standard)
- LR should be 2x-200x smaller than SFT LR

In [ ]:
import yaml
from pathlib import Path

Path("configs").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)

### Experiment A: Qwen3-4B DPO (after SFT)

In [ ]:
dpo_4b = {
    # Use merged SFT model as base
    "base_model": "./outputs/qwen3-4b-sft-merged",
    # OR: use HF hub if you pushed it
    # "base_model": "YOUR_USERNAME/qwen3-4b-sft",
    "model_type": "AutoModelForCausalLM",
    "tokenizer_type": "AutoTokenizer",

    "load_in_8bit": False,
    "load_in_4bit": True,

    # DPO-specific
    "rl": "dpo",
    "dpo_beta": 0.1,  # Beta 0.1+ for reference adherence

    "datasets": [
        {
            "path": "argilla/ultrafeedback-binarized-preferences",
            "type": "chat_template.argilla_dpo",
        }
    ],
    "val_set_size": 0.02,
    "output_dir": "./outputs/qwen3-4b-dpo-qlora",

    "sequence_len": 2048,
    "sample_packing": False,  # DPO typically doesn't use packing

    "adapter": "qlora",
    "lora_r": 32,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "lora_target_linear": True,

    "gradient_accumulation_steps": 8,
    "micro_batch_size": 1,
    "num_epochs": 1,  # DPO usually 1 epoch
    "optimizer": "adamw_bnb_8bit",
    "lr_scheduler": "cosine",
    "learning_rate": 5e-7,  # 10x smaller than SFT LR
    "warmup_ratio": 0.1,

    "bf16": "auto",
    "gradient_checkpointing": True,
    "flash_attention": True,

    "logging_steps": 1,
    "evals_per_epoch": 4,
    "saves_per_epoch": 1,
    "weight_decay": 0.0,

    "wandb_project": "axolotl-dpo-experiments",
    "wandb_name": "qwen3-4b-dpo-beta0.1-lr5e7",
}

with open("configs/qwen3_4b_dpo_qlora.yml", "w") as f:
    yaml.dump(dpo_4b, f, default_flow_style=False)

print("Wrote configs/qwen3_4b_dpo_qlora.yml")
print(yaml.dump(dpo_4b, default_flow_style=False))

### Experiment B: Qwen3-14B DPO (A100)

In [ ]:
import copy

dpo_14b = copy.deepcopy(dpo_4b)
dpo_14b["base_model"] = "./outputs/qwen3-14b-sft-merged"
dpo_14b["output_dir"] = "./outputs/qwen3-14b-dpo-qlora"
dpo_14b["sequence_len"] = 4096
dpo_14b["gradient_accumulation_steps"] = 16
dpo_14b["learning_rate"] = 3e-7  # More conservative for 14B
dpo_14b["wandb_name"] = "qwen3-14b-dpo-beta0.1-lr3e7"

with open("configs/qwen3_14b_dpo_qlora.yml", "w") as f:
    yaml.dump(dpo_14b, f, default_flow_style=False)

print("Wrote configs/qwen3_14b_dpo_qlora.yml")

### Beta Sweep (Qwen3-4B)

Sweep beta (0.1, 0.2, 0.5) to find optimal reference adherence.

In [ ]:
beta_sweep = [0.1, 0.2, 0.5]

for beta in beta_sweep:
    cfg = copy.deepcopy(dpo_4b)
    cfg["dpo_beta"] = beta
    cfg["wandb_name"] = f"qwen3-4b-dpo-beta{beta}-lr5e7"
    cfg["output_dir"] = f"./outputs/qwen3-4b-dpo-beta{beta}"
    fname = f"configs/qwen3_4b_dpo_beta_{beta}.yml"
    with open(fname, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print(f"  {fname} -> beta={beta}")

print(f"\nGenerated {len(beta_sweep)} beta sweep configs")

## 3. Train DPO

In [ ]:
CONFIG = "configs/qwen3_4b_dpo_qlora.yml"    # T4-safe
# CONFIG = "configs/qwen3_14b_dpo_qlora.yml"  # A100

In [ ]:
!accelerate launch -m axolotl.cli.train {CONFIG}

## 4. Evaluate DPO Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = "./outputs/qwen3-4b-sft-merged"
dpo_adapter = "./outputs/qwen3-4b-dpo-qlora"

tokenizer = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.bfloat16, device_map="auto")
model = PeftModel.from_pretrained(model, dpo_adapter)

test_prompts = [
    "What are the pros and cons of nuclear energy?",
    "Write a Python function to merge two sorted lists.",
    "Explain quantum computing to a 10-year-old.",
]

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
    response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Q: {prompt}")
    print(f"A: {response}")
    print("-" * 80)

## 5. Compare SFT vs DPO Side-by-Side

In [ ]:
# Load SFT-only model for comparison
sft_model = AutoModelForCausalLM.from_pretrained(
    "./outputs/qwen3-4b-sft-merged", torch_dtype=torch.bfloat16, device_map="auto"
)
sft_tokenizer = AutoTokenizer.from_pretrained("./outputs/qwen3-4b-sft-merged")

prompt = "Should I learn Rust or Go for systems programming?"
messages = [{"role": "user", "content": prompt}]

# SFT response
inputs = sft_tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
with torch.no_grad():
    sft_out = sft_model.generate(inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
print("=== SFT ===")
print(sft_tokenizer.decode(sft_out[0][inputs.shape[1]:], skip_special_tokens=True))

# DPO response
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
with torch.no_grad():
    dpo_out = model.generate(inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
print("\n=== DPO ===")
print(tokenizer.decode(dpo_out[0][inputs.shape[1]:], skip_special_tokens=True))

## 6. Merge & Push

In [ ]:
!accelerate launch -m axolotl.cli.merge_lora {CONFIG} \
    --lora-model-dir ./outputs/qwen3-4b-dpo-qlora \
    --output-dir ./outputs/qwen3-4b-dpo-merged

In [ ]:
# Push to Hub
from huggingface_hub import HfApi

# api = HfApi()
# api.upload_folder(
#     folder_path="./outputs/qwen3-4b-dpo-merged",
#     repo_id="YOUR_USERNAME/qwen3-4b-sft-dpo",
#     repo_type="model",
# )